# Face Multi-Task Research -- Colab Runner

Trains and evaluates the multi-task age / dataset gender-label model from
**face-multitask-research** on a free Colab GPU, with all checkpoints and
outputs persisted to Google Drive so nothing is lost when the runtime
recycles.

> **Research/education only.** Predictions may be inaccurate, biased, or
> unreliable. Gender-related output reflects labels in the training
> dataset, not a determination of identity. See the repo's `README.md`
> and `docs/model_card.md` / `docs/data_card.md` for full ethical
> limitations before using this for anything beyond a research demo.

**How this notebook is organized:**
1. Mount Drive, get the code, install dependencies.
2. Restore any previous run from Drive (safe no-op on a first run).
3. Kaggle credentials + download/prepare data.
4. Train -> calibrate -> k-NN index -> evaluate.
5. Robustness + Grad-CAM + architecture report.
6. Browse results inline, with a Drive sync after every stage that
   produces new checkpoints/outputs.

The frontend (React) is **not** used here -- it isn't needed to reproduce
any research result, only for an interactive local/Docker demo.

### Step 0 -- confirm you have a GPU
Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4 is fine), then run the cell below.

In [ ]:
!nvidia-smi

### Step 1 -- mount Google Drive
This is where checkpoints/outputs/data splits persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUN_DIR = "/content/drive/MyDrive/face-multitask-research-runs"
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
print("Drive run directory:", DRIVE_RUN_DIR)

### Step 2 -- get the project code

Pick **one** of the two options below and run only that cell.

- **Option A (recommended): clone from GitHub.** Push this repo to a
  GitHub repo first (public or private -- for private repos, clone with a
  personal access token in the URL, or use `%cd` + `git clone` after
  `!git config` with credentials).
- **Option B: unzip a copy you uploaded to Drive.** Zip your local
  `face-multitask-research` folder, upload the zip to
  `MyDrive/face-multitask-research.zip`, then run Option B instead.

Either way the code ends up in the local (fast, ephemeral) Colab disk at
`/content/face-multitask-research` -- keeping code and the active dataset
on local disk (not directly on Drive) makes training much faster than
reading/writing many small files over the Drive mount.

In [ ]:
# ---- Option A: clone from GitHub ----
REPO_URL = "https://github.com/<you>/face-multitask-research.git"  # <-- EDIT THIS
CODE_DIR = "/content/face-multitask-research"

import os
if not os.path.exists(CODE_DIR):
    !git clone {REPO_URL} {CODE_DIR}
else:
    print(f"{CODE_DIR} already exists, skipping clone")
%cd {CODE_DIR}

In [ ]:
# ---- Option B: unzip a copy uploaded to Drive ----
# ZIP_PATH = "/content/drive/MyDrive/face-multitask-research.zip"  # <-- EDIT THIS
# CODE_DIR = "/content/face-multitask-research"
#
# import os
# os.makedirs(CODE_DIR, exist_ok=True)
# !unzip -oq "{ZIP_PATH}" -d {CODE_DIR}
# %cd {CODE_DIR}

### Step 3 -- install dependencies
`torch` is already preinstalled on Colab GPU runtimes; this mainly adds the project-specific packages (opencv-headless, scikit-learn, kaggle, fastapi, etc.).

In [ ]:
!pip install -q -r requirements.txt

### Step 4 -- Drive sync helpers

`sync_to_drive()` copies fresh results to Drive after each stage;
`restore_from_drive()` pulls a previous run back in at the start of a new
session so you can resume instead of starting over. Both are safe no-ops
if the source doesn't exist yet.

In [ ]:
import os, shutil

def sync_to_drive(*dirnames):
    for name in dirnames:
        src = os.path.join(CODE_DIR, name)
        dst = os.path.join(DRIVE_RUN_DIR, name)
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"synced: {src}  ->  {dst}")
        else:
            print(f"skipped (not found yet): {src}")

def restore_from_drive(*dirnames):
    for name in dirnames:
        src = os.path.join(DRIVE_RUN_DIR, name)
        dst = os.path.join(CODE_DIR, name)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"restored: {src}  ->  {dst}")
        else:
            print(f"nothing to restore yet: {src}")

### Step 5 -- restore a previous run (optional, safe to run every session)
Pulls back checkpoints / outputs / data split from Drive if this notebook has been run before. Skips `data/raw` by default since re-downloading is usually faster than copying a large image set over the Drive mount -- uncomment if you'd rather restore it too.

In [ ]:
restore_from_drive("checkpoints", "outputs", "data/splits")
# restore_from_drive("data/raw")

### Step 6 -- Kaggle credentials

Get an API token at https://www.kaggle.com/settings ("Create New Token").
Prefer Colab's **Secrets** (key icon in the left sidebar) over typing keys
into a cell, since cell inputs can end up saved in the notebook's output
history. With a secret named `KAGGLE_KEY` set, the commented block below
will pick it up automatically; otherwise the `getpass` prompts ask for it
interactively (nothing is echoed or persisted in the notebook).

In [ ]:
import os, getpass

# --- Preferred: Colab Secrets (key icon in the left sidebar) ---
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
# os.environ["KAGGLE_DATASET_SLUG"] = userdata.get("KAGGLE_DATASET_SLUG")

# --- Fallback: interactive prompt (key is masked, not saved in the notebook) ---
if "KAGGLE_USERNAME" not in os.environ:
    os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ")
if "KAGGLE_KEY" not in os.environ:
    os.environ["KAGGLE_KEY"] = getpass.getpass("Kaggle API key: ")
if "KAGGLE_DATASET_SLUG" not in os.environ:
    os.environ["KAGGLE_DATASET_SLUG"] = input("Kaggle dataset slug (e.g. jangedoo/utkface-new): ") or "jangedoo/utkface-new"

print("Kaggle credentials set for user:", os.environ["KAGGLE_USERNAME"])
print("Dataset slug:", os.environ["KAGGLE_DATASET_SLUG"])

### Step 7 -- download + prepare data
Downloads via the official Kaggle API (skipped automatically if `data/raw` already has files -- pass `--force` to re-download), then validates images and writes a deterministic, leakage-checked train/val/test split.

In [ ]:
!python scripts/download_kaggle_data.py
!python scripts/prepare_data.py
sync_to_drive("data/splits")

### Step 8 -- train

Trains the default configuration (`configs/model.yaml`: shared backbone +
adapters + fixed loss weights). Edit the `--set` overrides to try other
architectures (see `configs/experiments.yaml` for the full ablation
matrix), or run Step 8b for the whole suite in one go.

A GPU epoch on a few thousand 128px images typically takes well under a
minute; see `docs/reproducibility.md` for rough compute expectations.

In [ ]:
!python scripts/train.py --experiment-name multitask
sync_to_drive("checkpoints", "outputs")

### Step 8b (optional) -- full architecture ablation suite
Runs Experiments A-D from `configs/experiments.yaml` back to back, reusing the same split. This takes roughly 4x as long as the single run above -- skip it if you only care about one configuration.

In [ ]:
!python scripts/run_experiments.py
sync_to_drive("checkpoints", "outputs")

### Step 9 -- pick the checkpoint to evaluate

Defaults to the "best balanced score" checkpoint from Step 8. Change
`CHECKPOINT_NAME` if you ran Step 8b and want to evaluate a specific
experiment instead (e.g. `exp_c_shared_adapters_best_balanced_score.pt`).

In [ ]:
CHECKPOINT_NAME = "multitask_best_balanced_score.pt"  # <-- edit if needed
CHECKPOINT = f"checkpoints/{CHECKPOINT_NAME}"
print("Using checkpoint:", CHECKPOINT)

### Step 10 -- calibrate, build the k-NN index, evaluate
Split-conformal calibration on the validation set, a distance-weighted k-NN baseline in embedding space, then a full test-set evaluation comparing both.

In [ ]:
!python scripts/calibrate.py --checkpoint {CHECKPOINT}
!python scripts/build_knn_index.py --checkpoint {CHECKPOINT}
!python scripts/evaluate.py --checkpoint {CHECKPOINT} --compare-knn
sync_to_drive("outputs")

### Step 11 -- robustness, Grad-CAM, architecture report
Deterministic corruption robustness sweep, manual Grad-CAM overlays, gradient-interference / representation-similarity analysis, and the final auto-generated Markdown report.

In [ ]:
!python scripts/run_robustness.py --checkpoint {CHECKPOINT} --max-samples 300
!python scripts/generate_gradcam.py --checkpoint {CHECKPOINT}
!python scripts/generate_architecture_report.py --checkpoint {CHECKPOINT}
sync_to_drive("outputs", "docs")

### Step 12 -- browse results inline

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob("outputs/plots/*.png"))[:5]:
    print(path)
    display(Image(filename=path))

In [ ]:
for path in sorted(glob.glob("outputs/gradcam/*.png"))[:6]:
    print(path)
    display(Image(filename=path))

In [ ]:
with open("docs/architecture_analysis_generated.md") as f:
    print(f.read())

### Done

Everything under `checkpoints/` and `outputs/` (plus the generated
report in `docs/`) has been synced to
`MyDrive/face-multitask-research-runs/`. Next session, just re-run Steps
1-5 (mount Drive, get the code, install deps, restore from Drive) to pick
up exactly where you left off -- no need to retrain from scratch unless
you want to.

Remember: results depend entirely on the dataset, labels, and split used
-- see `docs/data_card.md` and `docs/model_card.md` before drawing any
conclusions from them.